In [2]:
import random
import math

# 1. Définition des villes (coordonnées x, y)
villes = {
    'A': (0, 0),
    'B': (1, 5),
    'C': (5, 2),
    'D': (6, 6),
    'E': (8, 3),
    'F': (3, 7)
}
noms_villes = list(villes.keys())

def distance(ville1, ville2):
    x1, y1 = villes[ville1]
    x2, y2 = villes[ville2]
    return math.hypot(x1 - x2, y1 - y2)

def longueur_chemin(chemin):
    totale = 0
    for i in range(len(chemin)):
        ville_actuelle = chemin[i]
        ville_suivante = chemin[(i + 1) % len(chemin)]
        totale += distance(ville_actuelle, ville_suivante)
    return totale

# 2. Initialisation de la population
def creer_population(taille_pop):
    population = []
    for _ in range(taille_pop):
        chemin = noms_villes[:]
        random.shuffle(chemin)
        population.append(chemin)
    return population

# 3. Fonction de fitness (on veut minimiser la distance, donc on maximise l'inverse)
def evaluer_population(population):
    # Trie la population de la plus petite distance (meilleure) à la plus grande
    return sorted(population, key=lambda chemin: longueur_chemin(chemin))

# 4. Sélection par tournoi
def selection_tournoi(population, taille_tournoi=3):
    tournoi = random.sample(population, taille_tournoi)
    return min(tournoi, key=lambda chemin: longueur_chemin(chemin))

# 5. Croisement (Crossover) ordonné pour éviter les doublons de villes
def croisement(parent1, parent2):
    taille = len(parent1)
    debut, fin = sorted(random.sample(range(taille), 2))
    
    enfant = [None] * taille
    # Copier une partie du parent 1
    enfant[debut:fin+1] = parent1[debut:fin+1]
    
    # Remplir le reste avec le parent 2
    pointeur_p2 = 0
    for i in range(taille):
        if enfant[i] is None:
            while parent2[pointeur_p2] in enfant:
                pointeur_p2 += 1
            enfant[i] = parent2[pointeur_p2]
    return enfant

# 6. Mutation (échange de deux villes dans le chemin)
def muter(chemin, taux_mutation=0.1):
    nouveau_chemin = chemin[:]
    for i in range(len(nouveau_chemin)):
        if random.random() < taux_mutation:
            j = random.randint(0, len(nouveau_chemin) - 1)
            nouveau_chemin[i], nouveau_chemin[j] = nouveau_chemin[j], nouveau_chemin[i]
    return nouveau_chemin

# 7. Algorithme Principal (Boucle Génétique)
def algo_genetique(taille_pop=100, generations=200, taux_mutation=0.1):
    population = creer_population(taille_pop)
    
    for _ in range(generations):
        population = evaluer_population(population)
        meilleur_actuel = population[0]
        
        # Elitisme : on garde les meilleurs, on renouvelle le reste
        nouvelle_generation = population[:10] 
        
        while len(nouvelle_generation) < taille_pop:
            p1 = selection_tournoi(population)
            p2 = selection_tournoi(population)
            enfant = croisement(p1, p2)
            enfant = muter(enfant, taux_mutation)
            nouvelle_generation.append(enfant)
            
        population = nouvelle_generation
        
    meilleur_chemin = evaluer_population(population)[0]
    return meilleur_chemin, longueur_chemin(meilleur_chemin)

# Lancer l'algorithme
meilleur_chemin, distance_minimale = algo_genetique()
print("Meilleur chemin trouvé :", meilleur_chemin)
print("Distance totale :", distance_minimale)

Meilleur chemin trouvé : ['C', 'A', 'B', 'F', 'D', 'E']
Distance totale : 23.242718041274223
